# 0. Install Seqeval

In [1]:
!git clone https://github.com/chakki-works/seqeval.git /tmp/seqeval_repo
!cp -r /tmp/seqeval_repo/seqeval /usr/local/lib/python3.13/dist-packages/
!rm -rf /tmp/seqeval_repo
!echo "Install Seq Eval completed"

Cloning into '/tmp/seqeval_repo'...
remote: Enumerating objects: 808, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 808 (delta 6), reused 3 (delta 3), pack-reused 798 (from 2)
Receiving objects: 100% (808/808), 185.39 KiB | 1.76 MiB/s, done.
Resolving deltas: 100% (461/461), done.
Install Seq Eval completed


# 1. Clone repository

In [2]:
!git clone -q https://github.com/ducbao210/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora.git
%cd Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora

/kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora


In [3]:
!pwd

/kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora


# 2. Dependencies

In [4]:
!pip install -q -r requirements-kaggle.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.1 MB/s eta 0:00:00


In [5]:
# Kaggle may ship an old optional torchao that newer PEFT rejects during LoRA setup.
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

def secret(name: str) -> str:
    try:
        return user_secrets.get_secret(name)
    except Exception as exc:
        print(f"Secret {name} is not available: {exc}")
        return ""

hf_token = secret("HF_TOKEN_NER")
checkpoint_repo = secret("CHECKPOINT_REPO")

with open('.env', 'w') as f:
    f.write(f"HF_TOKEN={hf_token}\n")
    f.write(f"HF_CHECKPOINT_REPO_ID={checkpoint_repo}\n")

print("Successfully created .env!")

Successfully created .env!


# 3. Download dataset

In [7]:
!python download_data.py

Downloaded: dev_word.conll, test_word.conll, train_word.conll


# 4. Training models

In [8]:
!python main.py \
  --mode train \
  --model all \
  --use_crf \
  --upload_checkpoint


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
config.json: 100%|█████████████████████████████| 557/557 [00:00<00:00, 2.33MB/s]
vocab.txt: 100%|█████████████████████████████| 895k/895k [00:00<00:00, 10.1MB/s]
bpe.codes: 100%|███████████████████████████| 1.14M/1.14M [00:00<00:00, 12.5MB/s]
tokenizer.json: 100%|██████████████████████| 3.13M/3.13M [00:00<00:00, 45.7MB/s]
2026-10-19 02:19:32 - model - INFO - Instantiating LSTM model (CRF=True)

 lstm_crf_True Parameters
  Total parameters:       19,776,972
  Trainable parameters:   19,776,972
  Frozen parameters:               0
  Trainable ratio:           100.00%

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...t

In [9]:
# knowledge distillation: Teacher - Phobert; Student - BiLSTM
!python main.py --mode distill --model bilstm --use_crf --upload_checkpoint


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 11236.91it/s]
2026-10-19 07:47:14 - model - INFO - vinai/phobert-base encoder: 14,175,744/134,998,272 parameters trainable; top blocks=[10, 11]
2026-10-19 07:47:17 - model - INFO - Instantiating Bidirectional LSTM model (CRF=True)

 student_distilled_bilstm_crf_True Parameters
  Total parameters:       20,353,484
  Trainable parameters:   20,353,484
  Frozen parameters:               0
  Trainable ratio:           100.00%

/usr/local/lib/python3.13/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

# 5. Evaluations

In [10]:
!python main.py \
  --mode evaluate \
  --model all \
  --use_crf

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-10-19 08:50:16 - model - INFO - Instantiating LSTM model (CRF=True)
  -> Saved predictions to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/predictions/lstm_crf_predictions.txt
  -> Saved metrics to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/metrics/lstm_crf_token_metrics.csv
  -> Saved confusion matrix to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/plots/lstm_crf_confusion_matrix.png
  -> Saved entity distribution to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/plots/lstm_crf_entity_distribution.png
2026-10-19 08:50:29 - model - INFO - Instantiating Bi

In [11]:
# knowledge distillation: Teacher - Phobert; Student - BiLSTM
!python main.py --mode evaluate --model bilstm --use_crf \
  --checkpoint results/checkpoints/bilstm_distilled_crf_best_model.pt

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-10-19 08:55:42 - model - INFO - Instantiating Bidirectional LSTM model (CRF=True)
  -> Saved predictions to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/predictions/bilstm_crf_predictions.txt
  -> Saved metrics to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/metrics/bilstm_crf_token_metrics.csv
  -> Saved confusion matrix to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/plots/bilstm_crf_confusion_matrix.png
  -> Saved entity distribution to /kaggle/working/Medical-Named-Entity-Recognition-for-Vietnamese-COVID-19-Corpora/results/plots/bilstm_crf_entity_distribution.png


# 6. Inference example

In [12]:
!python main.py \
  --mode infer \
  --model all \
  --use_crf \
  --infer_text "Từ ngày 24-7 đến ngày 31-7, bệnh nhân được mẹ là bà H.T.P (47 tuổi) đón về nhà ở phường Phước Hoà (bằng xe máy), không đi đâu, chỉ ra tạp hoá Phượng, chợ Vườn Lài, phường An Sơn cùng mẹ bán tạp hoá ở đây."

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-10-19 08:56:16 - model - INFO - Instantiating LSTM model (CRF=True)
Từ
ngày
24-7
đến
ngày
31-7 [B-DATE]
,
bệnh_nhân
được
mẹ
là
bà
H.T.P [B-NAME]
(
47 [B-AGE]
tuổi
)
đón
về
nhà
ở
phường [B-LOCATION]
Phước_Hòa [I-LOCATION]
(
bằng
xe_máy
)
,
không
đi
đâu
,
chỉ
ra
tạp
hóa_Phượng
,
chợ
Vườn_Lài
,
phường [B-LOCATION]
An_Sơn
cùng
mẹ
bán
tạp_hóa
ở
đây
.
2026-10-19 08:56:17 - model - INFO - Instantiating Bidirectional LSTM model (CRF=True)
Từ
ngày
24-7 [B-DATE]
đến
ngày
31-7 [B-DATE]
,
bệnh_nhân
được
mẹ
là
bà
H.T.P [B-NAME]
(
47 [B-AGE]
tuổi
)
đón
về
nhà
ở
phường [B-LOCATION]
Phước_Hòa [I-LOCATION]
(
bằng
xe_máy
)
,
không
đi
đâu
,
chỉ
ra
tạp
hóa_Phượng
,
chợ [B-LOCATION]
Vườn_Lài [I-LOCATION]
,
phường [B-LOCATION]
An_Sơn [I-LOCATION]
cù

In [13]:
# knowledge distillation: Teacher - Phobert; Student - BiLSTM
!python main.py --mode infer --model bilstm --use_crf \
  --checkpoint results/checkpoints/bilstm_distilled_crf_best_model.pt \
  --infer_text "Từ ngày 24-7 đến ngày 31-7, bệnh nhân được mẹ là bà H.T.P (47 tuổi) đón về nhà ở phường Phước Hoà (bằng xe máy), không đi đâu, chỉ ra tạp hoá Phượng, chợ Vườn Lài, phường An Sơn cùng mẹ bán tạp hoá ở đây."


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
2026-10-19 08:56:52 - model - INFO - Instantiating Bidirectional LSTM model (CRF=True)
Từ
ngày
24-7 [B-DATE]
đến
ngày
31-7
,
bệnh_nhân
được
mẹ
là
bà
H.T.P [B-NAME]
(
47 [B-AGE]
tuổi
)
đón
về
nhà
ở
phường [B-LOCATION]
Phước_Hòa [I-LOCATION]
(
bằng
xe_máy
)
,
không
đi
đâu
,
chỉ
ra
tạp [B-LOCATION]
hóa_Phượng
,
chợ [B-LOCATION]
Vườn_Lài [I-LOCATION]
,
phường [B-LOCATION]
An_Sơn [I-LOCATION]
cùng
mẹ
bán
tạp_hóa
ở
đây
.


# 7. Compress results

In [14]:
!rm -rf .env results/checkpoints/ && zip -qr results.zip results/ && echo "Compressed successfully"

Compressed successfully
